# PLFS Exploratory Data Analysis
**IIM Mumbai PPM — Assignment & Learning**

---

**Data File Used:** `PLFS_selected_sample.csv` (100,000 records)

**Full Dataset:** `PLFS_selected.csv` (1,148,634 records)

**Source:** [microdata.gov.in/NADA — PLFS Catalog](https://microdata.gov.in/NADA/index.php/catalog/PLFS/)

**Variables:** 21 columns — st, sex, age, marst, gedu_lvl, tedu_lvl, curr_att, sas, ind_sas, ocu_sas, wrk_365, evr_wrk, ern_reg, ern_self, tothrs_wrk, totadl_wrk, dur_unp, eff_pas, voc, voc_fld, voc_typ

**Derived Column:** `total_income = ern_reg + ern_self`

## 1. Setup & Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Upload file in Colab: Files > Upload, or use:
# from google.colab import files
# uploaded = files.upload()

# FILE USED: PLFS_selected_sample.csv (100K records)
# Change to PLFS_selected.csv for full dataset (1.1M records)
FILE = 'PLFS_selected_sample.csv'

df = pd.read_csv(FILE)
print(f'File: {FILE}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

## 2. Data Preparation & Mappings

In [ ]:
# Filter to Male and Female only
df = df[df['sex'].isin([1, 2])].copy()

# Code mappings
SEX_MAP = {1: 'Male', 2: 'Female'}
MARST_MAP = {1: 'Never Married', 2: 'Currently Married', 3: 'Widowed', 4: 'Divorced/Separated'}
EDU_MAP = {1: 'Not literate', 2: 'Literate (no school)', 3: 'Below Primary', 4: 'Primary',
           5: 'Middle', 6: 'Secondary', 7: 'Higher Secondary', 8: 'Diploma', 10: 'Graduate',
           11: 'PG & above', 12: 'Graduate (Technical)', 13: 'PG (Technical)'}
SAS_MAP = {11: 'Self-employed (own account)', 12: 'Self-employed (employer)',
           21: 'Helper in HH enterprise', 31: 'Regular wage/salaried',
           41: 'Casual labour (public)', 51: 'Casual labour (other)'}

df['sex_label'] = df['sex'].map(SEX_MAP)
df['marst_label'] = df['marst'].map(MARST_MAP)
df['edu_label'] = df['gedu_lvl'].map(EDU_MAP)
df['sas_label'] = df['sas'].map(SAS_MAP)

# Derived columns
df['total_income'] = df['ern_reg'] + df['ern_self']
df['age_group'] = pd.cut(df['age'], bins=[0, 14, 17, 24, 29, 34, 44, 54, 64, 100],
                         labels=['0-14', '15-17', '18-24', '25-29', '30-34', '35-44', '45-54', '55-64', '65+'])

# Working age subset (15+)
df15 = df[df['age'] >= 15].copy()
df15['employed'] = df15['sas'].notna()
df15['in_lf'] = df15['employed'] | (df15['dur_unp'].notna())
df15['unemployed'] = (~df15['employed']) & (df15['dur_unp'].notna())

print(f'After filtering: {len(df):,} records')
print(f'Working age (15+): {len(df15):,} records')

## 3. Descriptive Statistics — Age

In [ ]:
print('=== AGE DESCRIPTIVE STATISTICS ===')
print(f'Count:    {len(df):,}')
print(f'Mean:     {df["age"].mean():.1f}')
print(f'Median:   {df["age"].median():.1f}')
print(f'Mode:     {df["age"].mode()[0]}')
print(f'Std Dev:  {df["age"].std():.1f}')
print(f'Min:      {df["age"].min()}')
print(f'Max:      {df["age"].max()}')
print(f'Q1 (25%): {df["age"].quantile(0.25):.0f}')
print(f'Q3 (75%): {df["age"].quantile(0.75):.0f}')
print(f'IQR:      {df["age"].quantile(0.75) - df["age"].quantile(0.25):.0f}')
print(f'Skewness: {df["age"].skew():.2f}')
print(f'Kurtosis: {df["age"].kurtosis():.2f}')
print()
print('--- By Gender ---')
for sex, label in [(1, 'Male'), (2, 'Female')]:
    s = df[df['sex'] == sex]['age']
    print(f'{label}: Mean={s.mean():.1f}, Median={s.median():.0f}, Std={s.std():.1f}')

## 4. KDE Density Plot — Age by Gender

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df[df['sex'] == 1]['age'].plot.kde(ax=ax, color='#2563eb', lw=2.5, label='Male')
df[df['sex'] == 2]['age'].plot.kde(ax=ax, color='#be185d', lw=2.5, label='Female')
df['age'].plot.kde(ax=ax, color='#1a1a2e', lw=2, ls='--', label='Overall')
ax.set_title('Kernel Density Plot — Age Distribution by Gender', fontsize=14, fontweight='bold')
ax.set_xlabel('Age'); ax.set_ylabel('Density'); ax.set_xlim(0, 100)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.show()

## 5. Age Quartile — Box Plot by Gender

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
data_m = df[df['sex'] == 1]['age']
data_f = df[df['sex'] == 2]['age']
bp = ax.boxplot([data_m, data_f], labels=['Male', 'Female'], patch_artist=True, showfliers=False)
bp['boxes'][0].set_facecolor('#2563eb'); bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('#be185d'); bp['boxes'][1].set_alpha(0.6)
ax.set_title('Age Quartile Distribution by Gender', fontsize=14, fontweight='bold')
ax.set_ylabel('Age (years)'); ax.grid(axis='y', alpha=0.3)

for i, (data, label) in enumerate([(data_m, 'Male'), (data_f, 'Female')]):
    q1, med, q3 = data.quantile([0.25, 0.5, 0.75])
    ax.text(i+1.3, q1, f'Q1={q1:.0f}', fontsize=9)
    ax.text(i+1.3, med, f'Med={med:.0f}', fontsize=9, fontweight='bold')
    ax.text(i+1.3, q3, f'Q3={q3:.0f}', fontsize=9)
plt.show()

## 6. Frequency Distribution — SAS (Status of Activity)

In [ ]:
print('Frequency distribution for sas column (including missing values):')
freq = df['sas'].value_counts(dropna=False).sort_index()
print(freq)
print(f"\nNumber of missing values in 'sas' column: {df['sas'].isna().sum():,}")
print(f"Percentage missing: {df['sas'].isna().mean()*100:.1f}%")

print('\n--- With Labels ---')
for code in [11, 12, 21, 31, 41, 51]:
    count = (df['sas'] == code).sum()
    pct = count / len(df) * 100
    print(f'{code} ({SAS_MAP[code]}): {count:,} ({pct:.1f}%)')
missing = df['sas'].isna().sum()
print(f'NaN (Missing / Not in LF): {missing:,} ({missing/len(df)*100:.1f}%)')

## 7. Gender Distribution — Donut Chart

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
sex_counts = df['sex_label'].value_counts()
ax.pie(sex_counts, labels=sex_counts.index, autopct='%1.1f%%',
       colors=['#2563eb', '#be185d'], startangle=90,
       wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2))
ax.set_title('Gender Distribution', fontsize=14, fontweight='bold')
plt.show()

## 8. Education Level — Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
edu_order = [1,2,3,4,5,6,7,8,10,11,12,13]
edu_counts = df['gedu_lvl'].value_counts().reindex(edu_order).fillna(0)
edu_labels = [EDU_MAP.get(e, str(e)) for e in edu_order]
ax.bar(edu_labels, edu_counts.values, color=plt.cm.viridis(np.linspace(0.2, 0.9, len(edu_order))), edgecolor='white')
ax.set_title('Education Level Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.xticks(rotation=40, ha='right')
plt.show()

## 9. LFPR by Gender — Bar Chart

In [ ]:
lfpr_gender = df15.groupby('sex_label')['in_lf'].mean() * 100
print('LFPR by Gender:')
print(lfpr_gender.round(1))

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(lfpr_gender.index, lfpr_gender.values, color=['#2563eb', '#be185d'], width=0.5)
for b, v in zip(bars, lfpr_gender.values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')
ax.set_title('Labour Force Participation Rate by Gender', fontsize=14, fontweight='bold')
ax.set_ylabel('LFPR (%)')
plt.show()

## 10. LFPR by Age Group & Gender — Multi-line

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for sex, color in [('Male', '#2563eb'), ('Female', '#be185d')]:
    sub = df15[df15['sex_label'] == sex]
    lfpr = sub.groupby('age_group', observed=False)['in_lf'].mean() * 100
    ax.plot(lfpr.index.astype(str), lfpr.values, 'o-', color=color, lw=2.5, label=sex)
ax.set_title('LFPR by Age Group & Gender', fontsize=14, fontweight='bold')
ax.set_ylabel('LFPR (%)'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.show()

## 11. Unemployment Rate by Age Group

In [ ]:
ur_age = df15.groupby('age_group', observed=False).apply(
    lambda x: x['unemployed'].sum() / x['in_lf'].sum() * 100 if x['in_lf'].sum() > 0 else 0,
    include_groups=False)
print('UR by Age Group:')
print(ur_age.round(1))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#dc2626' if v > 5 else '#f59e0b' if v > 2 else '#16a34a' for v in ur_age.values]
bars = ax.bar(ur_age.index.astype(str), ur_age.values, color=colors, width=0.6, edgecolor='white')
for b, v in zip(bars, ur_age.values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2, f'{v:.1f}%', ha='center', fontweight='bold')
ax.set_title('Unemployment Rate by Age Group', fontsize=14, fontweight='bold')
ax.set_ylabel('UR (%)')
plt.show()

## 12. UR by Education Level

In [ ]:
ur_edu = df15.groupby('edu_label').apply(
    lambda x: x['unemployed'].sum() / x['in_lf'].sum() * 100 if x['in_lf'].sum() > 0 else 0,
    include_groups=False).sort_values()
print('UR by Education:')
print(ur_edu.round(1))

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(ur_edu)))
ax.barh(ur_edu.index, ur_edu.values, color=colors, height=0.6)
ax.set_title('Unemployment Rate by Education Level', fontsize=14, fontweight='bold')
ax.set_xlabel('UR (%)')
for i, v in enumerate(ur_edu.values):
    ax.text(v + 0.2, i, f'{v:.1f}%', va='center', fontweight='bold')
plt.show()

## 13. Total Income — Descriptive Statistics & Outliers

In [ ]:
inc = df[df['total_income'] > 0]['total_income']

print('=== TOTAL INCOME DESCRIPTIVE STATISTICS ===')
print(f'Earners:    {len(inc):,}')
print(f'Mean:       Rs {inc.mean():,.0f}')
print(f'Median:     Rs {inc.median():,.0f}')
print(f'Std Dev:    Rs {inc.std():,.0f}')
print(f'Min:        Rs {inc.min():,.0f}')
print(f'Max:        Rs {inc.max():,.0f}')
print(f'Q1 (25%):   Rs {inc.quantile(0.25):,.0f}')
print(f'Q3 (75%):   Rs {inc.quantile(0.75):,.0f}')
print(f'IQR:        Rs {inc.quantile(0.75) - inc.quantile(0.25):,.0f}')
print(f'Skewness:   {inc.skew():.2f}')

q1 = inc.quantile(0.25)
q3 = inc.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
outliers = inc[inc > upper]
print(f'\n--- OUTLIER DETECTION (IQR Method) ---')
print(f'Upper Fence: Rs {upper:,.0f}')
print(f'Outliers:    {len(outliers):,} ({len(outliers)/len(inc)*100:.1f}%)')
print(f'Outlier Range: Rs {outliers.min():,.0f} - Rs {outliers.max():,.0f}')
print(f'Outlier Mean:  Rs {outliers.mean():,.0f}')

## 14. Income Box Plot with Outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot with outliers
ax = axes[0]
bp = ax.boxplot([inc], labels=['Total Income'], patch_artist=True, showfliers=True,
                flierprops=dict(marker='o', markersize=2, alpha=0.3, color='#dc2626'))
bp['boxes'][0].set_facecolor('#2563eb'); bp['boxes'][0].set_alpha(0.6)
ax.axhline(upper, color='#dc2626', ls='--', lw=1.5, label=f'Upper Fence: Rs {upper:,.0f}')
ax.set_title('Income Box Plot with Outliers', fontsize=13, fontweight='bold')
ax.set_ylabel('Monthly Income (Rs)'); ax.legend(fontsize=9)

# Histogram highlighting outliers
ax2 = axes[1]
ax2.hist(inc[inc <= upper], bins=50, color='#2563eb', alpha=0.7, edgecolor='white', label='Normal')
ax2.hist(inc[inc > upper], bins=30, color='#dc2626', alpha=0.7, edgecolor='white', label=f'Outliers ({len(outliers):,})')
ax2.axvline(upper, color='#dc2626', ls='--', lw=2)
ax2.set_title('Income Distribution — Outliers Highlighted', fontsize=13, fontweight='bold')
ax2.set_xlabel('Monthly Income (Rs)'); ax2.set_ylabel('Frequency'); ax2.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 15. Earnings by Gender — Box Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
data_m = inc[df.loc[inc.index, 'sex'] == 1].clip(upper=inc.quantile(0.95))
data_f = inc[df.loc[inc.index, 'sex'] == 2].clip(upper=inc.quantile(0.95))
bp = ax.boxplot([data_m, data_f], labels=['Male', 'Female'], patch_artist=True, showfliers=False)
bp['boxes'][0].set_facecolor('#2563eb'); bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('#be185d'); bp['boxes'][1].set_alpha(0.6)
ax.set_title('Earnings by Gender (Box Plot)', fontsize=14, fontweight='bold')
ax.set_ylabel('Monthly Earnings (Rs)')
plt.show()

print(f'Male Median:   Rs {data_m.median():,.0f}')
print(f'Female Median: Rs {data_f.median():,.0f}')
print(f'Gender Gap:    {abs(data_m.median()-data_f.median())/data_m.median()*100:.1f}%')

## 16. Employment Status — Stacked Bar by Gender

In [ ]:
sas_gender = pd.crosstab(df15['sex_label'], df15['sas_label'], normalize='index') * 100
print('Employment Status by Gender (%):')
print(sas_gender.round(1))

fig, ax = plt.subplots(figsize=(12, 6))
sas_gender.plot(kind='barh', stacked=True, ax=ax, colormap='tab10', edgecolor='white')
ax.set_title('Employment Status by Gender (Stacked)', fontsize=14, fontweight='bold')
ax.set_xlabel('Percentage (%)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## 17. Earnings Histogram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(inc.clip(upper=inc.quantile(0.95)), bins=50, color='#1a1a2e', alpha=0.7, edgecolor='white')
ax.axvline(inc.median(), color='#dc2626', ls='--', lw=2, label=f'Median: Rs {inc.median():,.0f}')
ax.axvline(inc.mean(), color='#f59e0b', ls='--', lw=2, label=f'Mean: Rs {inc.mean():,.0f}')
ax.set_title('Earnings Distribution (Clipped at 95th percentile)', fontsize=14, fontweight='bold')
ax.set_xlabel('Monthly Earnings (Rs)'); ax.set_ylabel('Frequency'); ax.legend()
plt.show()

## 18. LFPR by Marital Status & Gender

In [ ]:
lfpr_marst = df15.groupby(['marst_label', 'sex_label'])['in_lf'].mean() * 100
lfpr_marst_pivot = lfpr_marst.unstack()
print('LFPR by Marital Status & Gender:')
print(lfpr_marst_pivot.round(1))

fig, ax = plt.subplots(figsize=(10, 5))
lfpr_marst_pivot.plot(kind='bar', ax=ax, color=['#be185d', '#2563eb'], edgecolor='white', width=0.7)
ax.set_title('LFPR by Marital Status & Gender', fontsize=14, fontweight='bold')
ax.set_ylabel('LFPR (%)'); plt.xticks(rotation=30, ha='right')
ax.legend(title='Gender')
plt.show()

## 19. Work Hours Distribution

In [ ]:
hrs = df15['tothrs_wrk']
hrs = hrs[hrs > 0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(hrs, bins=40, color='#7c3aed', alpha=0.7, edgecolor='white')
ax.axvline(hrs.median(), color='#dc2626', ls='--', lw=2, label=f'Median: {hrs.median():.0f} hrs')
ax.set_title('Weekly Work Hours Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Hours per week'); ax.set_ylabel('Frequency'); ax.legend()
plt.show()

## 20. Age Distribution — Histogram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['age'], bins=50, color='#1a1a2e', alpha=0.7, edgecolor='white', density=True)
ax.axvline(df['age'].median(), color='#dc2626', ls='--', lw=2, label=f'Median: {df["age"].median():.0f}')
ax.axvline(df['age'].mean(), color='#f59e0b', ls='--', lw=2, label=f'Mean: {df["age"].mean():.1f}')
ax.set_title('Age Distribution (Histogram)', fontsize=14, fontweight='bold')
ax.set_xlabel('Age'); ax.set_ylabel('Density'); ax.legend()
plt.show()

## 21. Average Total Income by Gender
Filter: tothrs_wrk > 0 AND total_income > 0

In [ ]:
sub = df[(df['tothrs_wrk'] > 0) & (df['total_income'] > 0)]
print(f'Records: {len(sub):,}')
print()
avg = sub.groupby('sex')['total_income'].agg(['mean', 'median', 'count'])
avg.index = ['Male', 'Female']
avg.columns = ['Mean Income', 'Median Income', 'Count']
print(avg.round(0))
gap = (avg.loc['Male','Mean Income'] - avg.loc['Female','Mean Income']) / avg.loc['Male','Mean Income'] * 100
print(f'\nGender Gap (Mean): {gap:.1f}%')
gap_med = (avg.loc['Male','Median Income'] - avg.loc['Female','Median Income']) / avg.loc['Male','Median Income'] * 100
print(f'Gender Gap (Median): {gap_med:.1f}%')

## 22. Scatter: Age vs Income by Gender
Filter: tothrs_wrk > 0, total_income > 0

In [ ]:
from matplotlib.lines import Line2D
sub = df[(df['tothrs_wrk'] > 0) & (df['total_income'] > 0)]
sample = sub.sample(min(15000, len(sub)), random_state=42)
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2a9d8f' if s == 1 else '#e76f51' for s in sample['sex']]
ax.scatter(sample['age'], sample['total_income'], c=colors, alpha=0.4, s=20, edgecolors='none')
ax.set_title('Scatter Plot of Age vs. Total Income by Gender', fontsize=14, fontweight='bold')
ax.set_xlabel('Age'); ax.set_ylabel('Total Income')
ax.legend(handles=[Line2D([0],[0],marker='o',color='w',markerfacecolor='#2a9d8f',markersize=8,label='Male'),
                    Line2D([0],[0],marker='o',color='w',markerfacecolor='#e76f51',markersize=8,label='Female')], title='Gender')
ax.grid(alpha=0.2)
plt.show()

## 23. Scatter: Work Hours vs Income by Gender

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2a9d8f' if s == 1 else '#e76f51' for s in sample['sex']]
ax.scatter(sample['tothrs_wrk'], sample['total_income'], c=colors, alpha=0.4, s=20, edgecolors='none')
corr = sub['tothrs_wrk'].corr(sub['total_income'])
ax.set_title('Scatter Plot of Total Hours Worked vs. Total Income by Gender', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Hours Worked per Week'); ax.set_ylabel('Total Income')
ax.text(0.02, 0.95, f'n={len(sub):,} | Corr: {corr:.3f}', transform=ax.transAxes, fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.legend(handles=[Line2D([0],[0],marker='o',color='w',markerfacecolor='#2a9d8f',markersize=8,label='Male'),
                    Line2D([0],[0],marker='o',color='w',markerfacecolor='#e76f51',markersize=8,label='Female')], title='Gender')
ax.grid(alpha=0.2)
plt.show()

## 24. Worker Age Distribution (tothrs_wrk > 0) by Gender

In [ ]:
workers = df[(df['tothrs_wrk'] > 0)]
fig, ax = plt.subplots(figsize=(10, 5))
workers[workers['sex']==1]['age'].plot.kde(ax=ax, color='#2563eb', lw=2.5, label=f'Male (n={len(workers[workers["sex"]==1]):,})')
workers[workers['sex']==2]['age'].plot.kde(ax=ax, color='#be185d', lw=2.5, label=f'Female (n={len(workers[workers["sex"]==2]):,})')
ax.set_title('Age Distribution of Workers (tothrs_wrk > 0)', fontsize=14, fontweight='bold')
ax.set_xlabel('Age'); ax.set_ylabel('Density'); ax.set_xlim(10, 80)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.show()

## 25. Income/Hours Ratio by Education & Gender (tedu_lvl=1)

In [ ]:
sub2 = df[(df['tothrs_wrk'] > 0) & (df['total_income'] > 0) & (df['tedu_lvl'] == 1)]
agg = sub2.groupby(['gedu_lvl', 'sex']).agg(total_income_sum=('total_income', 'sum'), tothrs_wrk_sum=('tothrs_wrk', 'sum')).reset_index()
agg['ratio'] = agg['total_income_sum'] / (agg['tothrs_wrk_sum'] * 4)
male = agg[agg['sex'] == 1].sort_values('gedu_lvl')
female = agg[agg['sex'] == 2].sort_values('gedu_lvl')
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(male['gedu_lvl'], male['ratio'], c='blue', s=80, label='Male Hourly Income', marker='o')
ax.scatter(female['gedu_lvl'], female['ratio'], c='red', s=80, label='Female Hourly Income', marker='x', linewidths=2)
ax.set_title('Hourly Income by General Education Level (Technical Education Level = 1)', fontsize=13, fontweight='bold')
ax.set_xlabel('General Education Level'); ax.set_ylabel('Hourly Income (Total Monthly Income / Weekly Hours Worked * 4)')
ax.legend(); ax.grid(alpha=0.2)
ax.set_xticks(sorted(agg['gedu_lvl'].unique()))
plt.show()